In [14]:
#imports
import pandas as pd
import os, subprocess
from src.biotools.fasta_tools import *
from pathlib import Path
import json

In [4]:
# Selecting the accessions to run through Gromacs analysis

classifications = pd.read_csv('TMP_results/summary/summary_result.csv')[['accession','thermal_range']]
temperatures=pd.read_csv('top_hits/topt/top_hits_tp.csv')[['accession','tOPT']]

# Join classifications and temperatures
joined = pd.merge(
    classifications,
    temperatures,
    on='accession',
    validate='one_to_one'
)

# remove errors
joined = joined[joined['thermal_range'] !='ERROR']

# 3 psychrophiles
psychrophiles=joined[joined['thermal_range']=='psychrophile'].nsmallest(3, 'tOPT')
# 3 thermophiles
thermophiles=joined[joined['thermal_range']=='thermophile'].nlargest(3, 'tOPT')
# 3 random mesophiles
mesophiles=joined[joined['thermal_range']=='mesophile'].sample(n=3, random_state=10)

# 3 hottest from tOPT
hottest=joined.nlargest(3, "tOPT")
# 3 coldest from tOPT
coldest=joined.nsmallest(3, 'tOPT')
# Random sample
random=joined.sample(n=3, random_state=10)

mapping={
    'psychrophiles':list(psychrophiles['accession']),
    'thermophiles':list(thermophiles['accession']),
    'mesophiles':list(mesophiles['accession']),
    'hottest':list(hottest['accession']),
    'coldest':list(coldest['accession']),
    'random':list(random['accession']),
}


In [3]:
# Create directories
subprocess.run('mkdir uvsx/data/gmx_prep', shell=True)

# Create directories
for key in mapping.keys():
    subprocess.run(f'mkdir gmx_candidates/{key}/', shell=True)
    subprocess.run(f'mkdir gmx_candidates/{key}/sequences', shell=True)
    subprocess.run(f'mkdir gmx_candidates/{key}/structures', shell=True)


mkdir: uvsx/data/gmx_prep: File exists
mkdir: uvsx/data/gmx_prep/psychrophiles/: File exists
mkdir: uvsx/data/gmx_prep/psychrophiles/sequences: File exists
mkdir: uvsx/data/gmx_prep/thermophiles/: File exists
mkdir: uvsx/data/gmx_prep/thermophiles/sequences: File exists
mkdir: uvsx/data/gmx_prep/mesophiles/: File exists
mkdir: uvsx/data/gmx_prep/mesophiles/sequences: File exists
mkdir: uvsx/data/gmx_prep/hottest/: File exists
mkdir: uvsx/data/gmx_prep/hottest/sequences: File exists
mkdir: uvsx/data/gmx_prep/coldest/: File exists
mkdir: uvsx/data/gmx_prep/coldest/sequences: File exists
mkdir: uvsx/data/gmx_prep/random/: File exists
mkdir: uvsx/data/gmx_prep/random/sequences: File exists


In [28]:
# Creating Fastas for each protein
for i, j in mapping.items():
    for k in j:
        with open(f'gmx_candidates/{i}/sequences/{k}.fasta', 'w') as f:
            seq=ExtractSequence(f'{k}', 'top_hits/sequences/top_hits.fasta', header=True)
            f.write(f'{seq[0]}\n{seq[1]}')

Put sequences into alphafold3 server.
sequences are in gmx_candidates/structures under their respective accessions. e.g. AHZ95250.fasta > gmx_candidates/structures/ahz95250/ahz95250.pdb

In [18]:
# Converting the highest scoring structure to a PDB.

# Get list of all accs
accs=[item for sublist in mapping.values() for item in sublist]

# Get the best ranking structure for each protein
rankings={}
for acc in accs:
    acc=acc.lower()
    # Get summary files
    summaries = list(Path(f'gmx_candidates/structures/{acc}').glob('*summary*'))

    # Get ranking for each file
    scores={}
    for sum in summaries:
        with open(sum, 'r') as f:
            data = json.load(f)
            score=data["ranking_score"]
            scores[sum]=score

    # get file path with highest score
    best_file = max(scores, key=scores.get)

    # Get correct suffix
    best=str(best_file).split('.')[0].split('_')[-1]
    #Initialise correct target
    target=f'gmx_candidates/structures/{acc}/fold_{acc}_model_{best}.cif'


    # Run bash script to cnvert ifs to pdbs
    subprocess.run([
    "src/helpers/cif2pdb.sh",
    target,
    acc
])




Converting: YCB25778
Converting: UFK27161
Converting: YCQ78089
Converting: YP_874025
Converting: YP_004782219
Converting: AHZ95250
Converting: XCZ64124
Converting: WAX22921
Converting: UUV43823
Converting: YP_009791320
Converting: YP_004322308
Converting: XYR95885
Converting: XQO94789
Converting: YP_009151072
Converting: YP_007010273
Converting: YP_009213881
Converting: UVK61175
Converting: CAD5240622
